In [1]:
from google.colab import files
uploaded = files.upload()

Saving cleaned_data.csv to cleaned_data.csv


In [2]:
import pandas as pd
from io import BytesIO

# If you're uploading a CSV
df = pd.read_csv(BytesIO(uploaded['cleaned_data.csv']))
df

,Unnamed: 0,statement,status
0,0,oh my gosh,Anxiety
1,1,trouble sleeping confused mind restless heart ...,Anxiety
2,2,all wrong back off dear forward doubt stay in ...,Anxiety
3,3,ive shifted my focus to something else but im ...,Anxiety
4,4,im restless and restless its been a month now ...,Anxiety
...,...,...,...
38307,53038,nobody takes me seriously ive m dealt with dep...,Anxiety
38308,53039,selfishness i dont feel very good its like i d...,Anxiety
38309,53040,is there any way to sleep better i cant sleep ...,Anxiety
38310,53041,public speaking tips hi all i have to give a p...,Anxiety


In [3]:
df['statement'] = df['statement'].astype(str)
df['status'] = df['status'].astype(str)

In [4]:
!pip install transformers --quiet

In [5]:
import torch
import torch.nn as nn
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaTokenizer, RobertaModel
import torch.nn.functional as F

# ✅ Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [6]:
# Step 1: Save original class labels
original_classes = sorted(df['status'].unique())   # Strings: ["stress","anxiety","depression","normal"]

# Step 2: Create mapping
class_to_index = {cls: idx for idx, cls in enumerate(original_classes)}
index_to_class = {idx: cls for cls, idx in class_to_index.items()}

# Step 3: Convert df['status'] to integers
df['status'] = df['status'].map(class_to_index)

# Step 4: Number of classes
num_classes = len(original_classes)
print("Number of classes:", num_classes)
print("Index to class mapping:", index_to_class)

Number of classes: 4
Index to class mapping: {0: 'Anxiety', 1: 'Depression', 2: 'Normal', 3: 'Stress'}


In [7]:
class TextDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.texts = df['statement'].tolist()
        self.labels = df['status'].tolist()   # ← use integer labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = torch.tensor(self.labels[idx], dtype=torch.long)  # ← long, not float

        encoding = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'status': label
        }

In [8]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df['status']
)

In [10]:
from transformers import RobertaTokenizer, RobertaModel

tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [11]:
train_dataset = TextDataset(train_df, tokenizer)
test_dataset = TextDataset(test_df, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

In [12]:
class RobertaMultiClassScorer(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        self.roberta = RobertaModel.from_pretrained('roberta-base')
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(self.roberta.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        output = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        pooled = output.pooler_output
        x = self.dropout(pooled)
        return self.classifier(x)  # Raw logits

In [13]:
model = RobertaMultiClassScorer(num_classes=num_classes).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
loss_fn = nn.CrossEntropyLoss()

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [14]:
model.train()

for batch in train_loader:
    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    labels = batch['status'].to(device)  # shape: (batch_size)

    optimizer.zero_grad()

    logits = model(input_ids, attention_mask)  # shape: (batch_size, 4)

    loss = loss_fn(logits, labels)
    loss.backward()
    optimizer.step()

    print(f"Loss: {loss.item():.4f}")

Loss: 1.4030
Loss: 1.4257
Loss: 1.4048
Loss: 1.4130
Loss: 1.3951
Loss: 1.2589
Loss: 1.3590
Loss: 1.2526
Loss: 1.1906
Loss: 1.2937
Loss: 1.2137
Loss: 1.2764
Loss: 1.1363
Loss: 1.2251
Loss: 1.1375
Loss: 0.8759
Loss: 1.6989
Loss: 1.1028
Loss: 1.0381
Loss: 1.2814
Loss: 0.9399
Loss: 1.3177
Loss: 1.1312
Loss: 1.3911
Loss: 0.7553
Loss: 1.1864
Loss: 1.1751
Loss: 0.8228
Loss: 1.1549
Loss: 1.0324
Loss: 1.0441
Loss: 0.9901
Loss: 0.7953
Loss: 0.5951
Loss: 1.7093
Loss: 1.2317
Loss: 0.5664
Loss: 0.5351
Loss: 0.5571
Loss: 0.9629
Loss: 1.1185
Loss: 0.8234
Loss: 0.8447
Loss: 0.4648
Loss: 0.8412
Loss: 0.9312
Loss: 0.7871
Loss: 0.8850
Loss: 1.0568
Loss: 0.7316
Loss: 0.7516
Loss: 1.0314
Loss: 0.5007
Loss: 0.8602
Loss: 0.8877
Loss: 0.6042
Loss: 1.1489
Loss: 0.7088
Loss: 0.7400
Loss: 0.7428
Loss: 0.8080
Loss: 0.9509
Loss: 0.4824
Loss: 0.6680
Loss: 0.8395
Loss: 0.3913
Loss: 1.0907
Loss: 0.8084
Loss: 0.5265
Loss: 0.6373
Loss: 0.8908
Loss: 0.8204
Loss: 0.9272
Loss: 1.1647
Loss: 0.4174
Loss: 0.4494
Loss: 0.6632

In [15]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['status'].to(device)

        logits = model(input_ids, attention_mask)
        preds = torch.argmax(logits, dim=1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

accuracy = correct / total
print(f"\n✅ Test Accuracy: {accuracy * 100:.2f}%")


✅ Test Accuracy: 91.74%


In [16]:
def predict(text, model, tokenizer, device, index_to_class):
    model.eval()

    if not isinstance(text, str):
        text = str(text)

    with torch.no_grad():
        inputs = tokenizer(
            text,
            return_tensors='pt',
            truncation=True,
            padding=True,
            max_length=128
        )

        inputs = {k: v.to(device) for k, v in inputs.items()}

        logits = model(inputs['input_ids'], inputs['attention_mask'])

        probs = torch.softmax(logits, dim=1)
        probs = probs.squeeze().cpu()

        predicted_index = torch.argmax(probs).item()
        predicted_class = index_to_class[predicted_index]

        result = {
            index_to_class[i]: round(prob.item() * 100, 2)
            for i, prob in enumerate(probs)
        }

        return predicted_class, result

In [25]:
# Anxiety
text_input = "I'm sometimes worried something will go wrong, even when everything seems okay"

In [26]:
predicted_class, scores = predict(
    text_input, model, tokenizer, device, index_to_class
)

print(f"\nText: {text_input}")
print(f"\nPredicted Class: {predicted_class}")
print("\nClass Probabilities (%):")

for cls, score in scores.items():
    print(f"{cls}: {score}%")


Text: I'm sometimes worried something will go wrong, even when everything seems okay

Predicted Class: Anxiety

Class Probabilities (%):
Anxiety: 76.28%
Depression: 4.16%
Normal: 15.65%
Stress: 3.91%


In [19]:
# Save model to a file
model_path = "roberta_multiclass_model.pth"
torch.save(model.state_dict(), model_path)
print("✅ Model saved.")

✅ Model saved.


In [20]:
from google.colab import files
files.download(model_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>